# HiRISE Dataset EDA

Explores class distribution, sample images, and inter-class visual similarity.
Identifies confusable terrain class pairs (bright_dunes↔dunes, spiders↔swiss_cheese)
that motivate the hard negative mining strategy.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter

from src.dataset import build_class_index

DATA_ROOT = '../data/hirise'
class_index = build_class_index(DATA_ROOT)
print('Classes found:', sorted(class_index.keys()))
print('Counts:', {k: len(v) for k, v in class_index.items()})

In [ ]:
# Class distribution bar chart
counts = {k: len(v) for k, v in class_index.items()}
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(counts.keys(), counts.values())
ax.set_xlabel('Terrain class')
ax.set_ylabel('Image count')
ax.set_title('HiRISE Class Distribution')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Sample images per class
import random
classes = sorted(class_index.keys())
n_cols = 4
fig, axes = plt.subplots(len(classes), n_cols, figsize=(n_cols * 2.5, len(classes) * 2.5))
for row, cls in enumerate(classes):
    samples = random.sample(class_index[cls], min(n_cols, len(class_index[cls])))
    for col, path in enumerate(samples):
        ax = axes[row, col]
        ax.imshow(Image.open(path).convert('L'), cmap='gray')
        ax.axis('off')
        if col == 0:
            ax.set_title(cls, fontsize=8, loc='left')
plt.suptitle('Sample Images per Class', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Mean image per class — visualize intra-class vs inter-class appearance
from torchvision import transforms
import torch

transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

class_means = {}
for cls, paths in class_index.items():
    imgs = [transform(Image.open(p).convert('L')) for p in paths[:100]]
    class_means[cls] = torch.stack(imgs).mean(0).squeeze().numpy()

fig, axes = plt.subplots(1, len(class_means), figsize=(len(class_means) * 2.2, 2.5))
for ax, (cls, mean_img) in zip(axes, class_means.items()):
    ax.imshow(mean_img, cmap='gray')
    ax.set_title(cls, fontsize=7)
    ax.axis('off')
plt.suptitle('Per-class Mean Image (N=100)')
plt.tight_layout()
plt.show()

In [ ]:
# Pairwise pixel-space cosine similarity between class means
# High similarity between two classes → they are good candidates for hard negative mining

classes_ordered = sorted(class_means.keys())
mean_vecs = np.array([class_means[c].flatten() for c in classes_ordered])
mean_vecs /= np.linalg.norm(mean_vecs, axis=1, keepdims=True) + 1e-9
sim_matrix = mean_vecs @ mean_vecs.T

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(sim_matrix, xticklabels=classes_ordered, yticklabels=classes_ordered,
            annot=True, fmt='.2f', cmap='coolwarm', vmin=0.7, vmax=1.0, ax=ax)
ax.set_title('Pixel-space cosine similarity between class means\n(high = visually confusable)')
plt.tight_layout()
plt.show()
print('Highest off-diagonal pairs indicate best candidates for hard negative mining.')